In [1]:
import os
import numpy as np
import pandas as pd
from itertools import combinations
from collections import defaultdict

os.makedirs("tables", exist_ok=True)
np.set_printoptions(suppress=True, precision=4)

CSV = "diabetes.csv"
df = pd.read_csv(CSV)
FEATURES = [c for c in df.columns if c != "Outcome"]
print(df.shape, "признаки:", FEATURES)

(768, 9) признаки: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


In [2]:
medians_full = df[FEATURES].median(numeric_only=True)
Xbin = pd.DataFrame({c: (df[c] <= medians_full[c]).astype(int) for c in FEATURES})


def contingency_2x2(a: pd.Series, b: pd.Series):
    a = a.astype(bool)
    b = b.astype(bool)
    aa = int((a & b).sum())
    bb = int((a & (~b)).sum())
    cc = int(((~a) & b).sum())
    dd = int(((~a) & (~b)).sum())
    return aa, bb, cc, dd


def yule_y(a, b, c, d):
    ad, bc = a * d, b * c
    if ad == 0 and bc == 0:
        return 0.0
    return (np.sqrt(ad) - np.sqrt(bc)) / (np.sqrt(ad) + np.sqrt(bc))


Y = np.zeros((len(FEATURES), len(FEATURES)))
for i, fi in enumerate(FEATURES):
    for j, fj in enumerate(FEATURES):
        a, b, c, d = contingency_2x2(Xbin[fi], Xbin[fj])
        Y[i, j] = yule_y(a, b, c, d)

sim_Y = pd.DataFrame(np.abs(Y), index=FEATURES, columns=FEATURES)

In [3]:
def mst_groups(sim: pd.DataFrame, target_m: int):
    cols = list(sim.index)
    p = len(cols)
    edges = []
    for i in range(p):
        for j in range(i + 1, p):
            w = max(float(sim.iloc[i, j]), float(sim.iloc[j, i]))
            edges.append((w, i, j))
    edges.sort(reverse=True, key=lambda t: t[0])
    parent = list(range(p))
    size = [1] * p

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    comps = p
    for w, i, j in edges:
        if comps <= target_m:
            break
        if union(i, j):
            comps -= 1

    groups = {}
    for i, name in enumerate(cols):
        r = find(i)
        groups.setdefault(r, []).append(name)
    components = [sorted(v) for v in groups.values()]
    return sorted(components, key=len, reverse=True)


def make_composites(components: list[list[str]]):
    X = df[FEATURES].to_numpy(dtype=float)
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    sigma[sigma == 0] = 1.0
    Z = (X - mu) / sigma
    names = []
    cols_z = []
    for gi, comp in enumerate(components, start=1):
        idx = [FEATURES.index(c) for c in comp]
        cols_z.append(Z[:, idx].mean(axis=1))
        names.append(f"G{gi}")
    Z_red = np.vstack(cols_z).T
    return Z_red, names



TARGET_M = 2  

components = mst_groups(sim_Y, TARGET_M)
Z_red, G_NAMES = make_composites(components)

print("Число композитов m =", Z_red.shape[1])
for i, comp in enumerate(components, start=1):
    print(f"  G{i}: {comp}")

Число композитов m = 2
  G1: ['Age', 'BMI', 'BloodPressure', 'Glucose', 'Insulin', 'Pregnancies', 'SkinThickness']
  G2: ['DiabetesPedigreeFunction']


In [4]:
def kmeans2_manual(Xr: np.ndarray, max_iter: int = 100, tol: float = 1e-6):
    c1 = Xr[0].copy()
    d2 = np.sum((Xr - c1) ** 2, axis=1)
    c2 = Xr[np.argmax(d2)].copy()
    centers = np.vstack([c1, c2])
    for _ in range(max_iter):
        dist1 = np.sqrt(np.sum((Xr - centers[0]) ** 2, axis=1))
        dist2 = np.sqrt(np.sum((Xr - centers[1]) ** 2, axis=1))
        labels = (dist2 < dist1).astype(int)
        new_c = centers.copy()
        for k in [0, 1]:
            pts = Xr[labels == k]
            if len(pts) > 0:
                new_c[k] = pts.mean(axis=0)
        if np.max(np.abs(new_c - centers)) < tol:
            centers = new_c
            break
        centers = new_c
    return labels.astype(int), centers


y_cluster, centers_km = kmeans2_manual(Z_red)
print("Размеры кластеров:", np.bincount(y_cluster, minlength=2))
y = y_cluster  

Размеры кластеров: [576 192]


In [5]:
SPLIT_MODE = "variational_glucose"  

if SPLIT_MODE == "stratified_y":
    order = np.argsort(y, kind="stable")
    split_desc = "стратификация по метке кластера y + чередование"
else:
    order = np.argsort(df["Glucose"].to_numpy(), kind="stable")
    split_desc = "вариационный ряд по Glucose + чередование"

work_idx = order[0::2]
ctrl_idx = order[1::2]

Z_work = Z_red[work_idx]
Z_ctrl = Z_red[ctrl_idx]
y_work = y[work_idx]
y_ctrl = y[ctrl_idx]

print("Разбиение:", split_desc)
print("Рабочая:", len(work_idx), "контрольная:", len(ctrl_idx))
print("y на рабочей:", np.bincount(y_work, minlength=2))
print("y на контроле:", np.bincount(y_ctrl, minlength=2))

Разбиение: вариационный ряд по Glucose + чередование
Рабочая: 384 контрольная: 384
y на рабочей: [290  94]
y на контроле: [286  98]


In [6]:
BIN_MODE = "global_mean"  

if BIN_MODE == "work_median":
    thr = np.median(Z_work, axis=0)
    bin_desc = "1 если значение <= медианы рабочей выборки по каждому композиту, иначе 0"
    Xb_work = (Z_work <= thr).astype(int)
    Xb_ctrl = (Z_ctrl <= thr).astype(int)
else:
    thr = Z_red.mean(axis=0)
    bin_desc = "1 если значение <= среднего по всей выборке по каждому композиту, иначе 0"
    Xb_full = (Z_red <= thr).astype(int)
    Xb_work = Xb_full[work_idx]
    Xb_ctrl = Xb_full[ctrl_idx]

m = Xb_work.shape[1]
print("BIN_MODE:", BIN_MODE)
print("Пороги:", dict(zip(G_NAMES, thr)))

BIN_MODE: global_mean
Пороги: {'G1': np.float64(6.534125092845973e-17), 'G2': np.float64(2.4517425127138875e-16)}


In [7]:
def build_sdnf(X_bin, y_loc, feat_mask=None):
    if feat_mask is not None:
        X_bin = X_bin[:, feat_mask]
    groups = defaultdict(list)
    for row, lbl in zip(X_bin, y_loc):
        groups[tuple(row)].append(int(lbl))
    lookup = {vec: (1 if sum(lbls) > len(lbls) / 2 else 0) for vec, lbls in groups.items()}
    default_class = 1 if y_loc.mean() >= 0.5 else 0
    return lookup, default_class


def apply_sdnf(X_bin, lookup, default_class, feat_mask=None):
    if feat_mask is not None:
        X_bin = X_bin[:, feat_mask]
    return np.array([lookup.get(tuple(row), default_class) for row in X_bin])


full_lookup, full_default = build_sdnf(Xb_work, y_work)
label = apply_sdnf(Xb_ctrl, full_lookup, full_default)

acc_full = (label == y_ctrl).mean()
print("Полная модель на композитах: совпадение с k-means на контроле:", f"{acc_full:.4f}")
print("Уникальных векторов на рабочей:", len(full_lookup), "из", 2**m)

rows = []
for r in range(1, m + 1):
    for subset in combinations(range(m), r):
        mask = list(subset)
        lu, dc = build_sdnf(Xb_work, y_work, feat_mask=mask)
        pred = apply_sdnf(Xb_ctrl, lu, dc, feat_mask=mask)
        acc_vs_label = (pred == label).mean()
        acc_vs_true = (pred == y_ctrl).mean()
        feat_str = "+".join(G_NAMES[i] for i in subset)
        rows.append({
            "n": r,
            "features": feat_str,
            "acc_vs_label": acc_vs_label,
            "acc_vs_true": acc_vs_true,
            "lookup_size": len(lu),
        })

resdf = pd.DataFrame(rows).sort_values(["acc_vs_label", "n"], ascending=[False, True])
print("\nТоп-10 по acc_vs_label:")
print(resdf.head(10).to_string(index=False))

out = f"tables/adequacy_composites_m{TARGET_M}.csv"
resdf.to_csv(out, index=False)
print("\nСохранено:", out)

Полная модель на композитах: совпадение с k-means на контроле: 0.8724
Уникальных векторов на рабочей: 4 из 4

Топ-10 по acc_vs_label:
 n features  acc_vs_label  acc_vs_true  lookup_size
 1       G2      1.000000     0.872396            2
 2    G1+G2      1.000000     0.872396            4
 1       G1      0.617188     0.744792            2

Сохранено: tables/adequacy_composites_m2.csv


In [8]:
outcome = df["Outcome"].to_numpy(dtype=int)

acc0 = (outcome == y).mean()
acc1 = (outcome == (1 - y)).mean()
print("Совпадение Outcome с k-means (без перестановки):", acc0)
print("Совпадение при инверсии меток кластера:", acc1)

Совпадение Outcome с k-means (без перестановки): 0.6510416666666666
Совпадение при инверсии меток кластера: 0.3489583333333333


In [ ]:
import math

def minterm_to_str(minterm, names):
    parts = []
    for val, name in zip(minterm, names):
        if val is None:
            continue
        if val == 1:
            parts.append(name)
        else:
            parts.append(f"!{name}")
    return " & ".join(parts) if parts else "1"


def minterms_can_merge(m1, m2):
    diff_pos = None
    for i in range(len(m1)):
        if m1[i] != m2[i]:
            if diff_pos is not None:
                return None
            diff_pos = i
    return diff_pos


def quine_mccluskey(minterms, n_vars):
    current = set()
    for m in minterms:
        current.add(tuple(m))
    all_prime = set()
    while current:
        used = set()
        new_implicants = set()
        current_list = sorted(
            current, key=lambda t: tuple((x if x is not None else -1) for x in t)
        )
        for i in range(len(current_list)):
            for j in range(i + 1, len(current_list)):
                diff_pos = minterms_can_merge(current_list[i], current_list[j])
                if diff_pos is not None:
                    merged = list(current_list[i])
                    merged[diff_pos] = None
                    new_implicants.add(tuple(merged))
                    used.add(current_list[i])
                    used.add(current_list[j])
        for impl in current:
            if impl not in used:
                all_prime.add(impl)
        current = new_implicants
    minterm_set = set(tuple(m) for m in minterms)
    prime_list = list(all_prime)

    def covers(impl, minterm):
        for k in range(len(impl)):
            if impl[k] is not None and impl[k] != minterm[k]:
                return False
        return True

    uncovered = set(minterm_set)
    selected = []
    while uncovered:
        best_impl = None
        best_count = 0
        for impl in prime_list:
            count = sum(1 for m in uncovered if covers(impl, m))
            if count > best_count:
                best_count = count
                best_impl = impl
        if best_impl is None or best_count == 0:
            break
        selected.append(best_impl)
        uncovered = {m for m in uncovered if not covers(best_impl, m)}
    return selected


def evaluate_dnf(row, implicants, names):
    for impl in implicants:
        match = True
        for k in range(len(impl)):
            if impl[k] is not None and impl[k] != row[names[k]]:
                match = False
                break
        if match:
            return 1
    return 0


def ks_stat(train_vals, test_vals):
    all_vals = np.sort(np.unique(np.concatenate([train_vals, test_vals])))
    d_max = 0.0
    for v in all_vals:
        f_train = np.mean(train_vals <= v)
        f_test = np.mean(test_vals <= v)
        d_max = max(d_max, abs(f_train - f_test))
    n1, n2 = len(train_vals), len(test_vals)
    d_crit = 1.36 * math.sqrt((n1 + n2) / (n1 * n2))
    return d_max, d_crit


outcome = df["Outcome"].to_numpy(dtype=int)

ks_rows = []
for i, gname in enumerate(G_NAMES):
    tr = Z_red[work_idx, i]
    te = Z_red[ctrl_idx, i]
    dn, dc = ks_stat(tr, te)
    ks_rows.append({
        "feature": gname,
        "D_n": round(dn, 4),
        "D_crit": round(dc, 4),
        "result": "ОК" if dn <= dc else "РАЗЛИЧАЮТСЯ",
    })
ks_df = pd.DataFrame(ks_rows)
ks_csv = f"tables/ks_composites_m{TARGET_M}.csv"
ks_df.to_csv(ks_csv, index=False, encoding="utf-8-sig")

train_bin_df = pd.DataFrame(Xb_work, columns=G_NAMES)
train_bin_df["Cluster"] = y_work
positive_train = train_bin_df[train_bin_df["Cluster"] == 1][G_NAMES]
unique_minterms = positive_train.drop_duplicates().values.tolist()
n_pos_train = int((y_work == 1).sum())

sdnf_terms = [minterm_to_str(m, G_NAMES) for m in unique_minterms]
sdnf_formula = " V\n    ".join(f"({t})" for t in sdnf_terms)

minimized = quine_mccluskey(unique_minterms, m)
min_terms_str = [minterm_to_str(list(impl), G_NAMES) for impl in minimized]
min_formula = " V\n    ".join(f"({t})" for t in min_terms_str)

train_logic_pred = train_bin_df[G_NAMES].apply(
    lambda row: evaluate_dnf(row, minimized, G_NAMES), axis=1
).values
train_acc = np.mean(train_logic_pred == y_work)
tr_tp = int(np.sum((train_logic_pred == 1) & (y_work == 1)))
tr_fn = int(np.sum((train_logic_pred == 0) & (y_work == 1)))
tr_fp = int(np.sum((train_logic_pred == 1) & (y_work == 0)))
tr_tn = int(np.sum((train_logic_pred == 0) & (y_work == 0)))

test_bin_df = pd.DataFrame(Xb_ctrl, columns=G_NAMES)
test_logic_pred = test_bin_df[G_NAMES].apply(
    lambda row: evaluate_dnf(row, minimized, G_NAMES), axis=1
).values
test_acc_vs_cluster = np.mean(test_logic_pred == y_ctrl)
test_tp = int(np.sum((test_logic_pred == 1) & (y_ctrl == 1)))
test_fn = int(np.sum((test_logic_pred == 0) & (y_ctrl == 1)))
test_fp = int(np.sum((test_logic_pred == 1) & (y_ctrl == 0)))
test_tn = int(np.sum((test_logic_pred == 0) & (y_ctrl == 0)))

outcome_ctrl = outcome[ctrl_idx]
cluster_vs_outcome_acc = np.mean(y_ctrl == outcome_ctrl)
logic_vs_outcome_acc = np.mean(test_logic_pred == outcome_ctrl)
agreement = test_acc_vs_cluster

if SPLIT_MODE == "variational_glucose":
    split_txt = "вариационный ряд по Glucose + чередование"
else:
    split_txt = "стратификация по метке кластера y + чередование"

bin_txt = (
    "1 если значение <= среднего, иначе 0"
    if BIN_MODE == "global_mean"
    else "1 если значение <= медианы рабочей выборки, иначе 0"
)

report_path = f"tables/logic_model_composites_m{TARGET_M}.txt"
sep = "=" * 60
with open(report_path, "w", encoding="utf-8") as f:
    f.write("ЛОГИЧЕСКАЯ МОДЕЛЬ (СДНФ и минимизация)\n")
    f.write(sep + "\n\n")
    f.write(f"Признаки: {', '.join(G_NAMES)}\n")
    f.write(f"Бинаризация: {bin_txt}\n\n")
    f.write(f"Разбиение: {split_txt}\n")
    f.write(f"  Рабочая: {len(work_idx)}, Контрольная: {len(ctrl_idx)}\n\n")
    f.write("Проверка Колмогорова-Смирнова (alpha=0.05):\n")
    for _, row in ks_df.iterrows():
        f.write(
            f"  {row['feature']}: D_n={row['D_n']}, D_crit={row['D_crit']} -> {row['result']}\n"
        )
    f.write("\n")
    f.write("ОБУЧАЮЩАЯ ЧАСТЬ:\n")
    f.write(f"  Строк с Cluster=1: {n_pos_train}\n")
    f.write(f"  Уникальных минтермов: {len(unique_minterms)}\n\n")
    f.write(f"СДНФ ({len(unique_minterms)} минтермов):\n")
    f.write(f"  {sdnf_formula}\n\n")
    f.write(f"Минимизированная ДНФ ({len(minimized)} импликант):\n")
    f.write(f"  {min_formula}\n\n")
    f.write("ПРОВЕРКА НА ОБУЧАЮЩЕЙ ВЫБОРКЕ:\n")
    f.write(f"  Логическая модель vs кластеризация: {train_acc:.4f}\n\n")
    f.write("ПРОВЕРКА НА ТЕСТОВОЙ ВЫБОРКЕ:\n")
    f.write(f"  Логическая модель vs кластеризация: {test_acc_vs_cluster:.4f}\n")
    f.write(f"  TP={test_tp}, FN={test_fn}, FP={test_fp}, TN={test_tn}\n\n")
    f.write("СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТЕ vs Outcome:\n")
    f.write(f"  K-means (стохастическая):  {cluster_vs_outcome_acc:.4f}\n")
    f.write(f"  Логическая (минимиз. ДНФ): {logic_vs_outcome_acc:.4f}\n")
    f.write(f"  Совпадение моделей между собой: {agreement:.4f}\n")

print("=" * 60)
print("ЛОГИЧЕСКАЯ МОДЕЛЬ (СДНФ и минимизация), композиты")
print("=" * 60)
print(f"КС по композитам (см. также {ks_csv}):")
print(ks_df.to_string(index=False))
print(f"\nОбучающая: Cluster=1 в {n_pos_train} строках, уникальных минтермов: {len(unique_minterms)}")
print(f"СДНФ: {len(unique_minterms)} минтермов; минимизированная ДНФ: {len(minimized)} импликант")
print(f"\nTrain: логика vs кластер {train_acc:.4f} | TP={tr_tp} FN={tr_fn} FP={tr_fp} TN={tr_tn}")
print(f"Test:  логика vs кластер {test_acc_vs_cluster:.4f} | TP={test_tp} FN={test_fn} FP={test_fp} TN={test_tn}")
print(f"Test vs Outcome: k-means {cluster_vs_outcome_acc:.4f}, логика {logic_vs_outcome_acc:.4f}, согласованность моделей {agreement:.4f}")
print(f"\nОтчёт сохранён: {report_path}")

ЛОГИЧЕСКАЯ МОДЕЛЬ (СДНФ и минимизация), композиты
КС по композитам (см. также tables/ks_composites_m2.csv):
feature    D_n  D_crit result
     G1 0.0677  0.0981     ОК
     G2 0.0312  0.0981     ОК

Обучающая: Cluster=1 в 94 строках, уникальных минтермов: 2
СДНФ: 2 минтермов; минимизированная ДНФ: 1 импликант

Train: логика vs кластер 0.8594 | TP=94 FN=0 FP=54 TN=236
Test:  логика vs кластер 0.8724 | TP=98 FN=0 FP=49 TN=237
Test vs Outcome: k-means 0.6458, логика 0.5859, согласованность моделей 0.8724

Отчёт сохранён: tables/logic_model_composites_m2.txt


In [ ]:
quine_mccluskey
ks_stat


TARGET_M = 4
components = mst_groups(sim_Y, TARGET_M)
Z_red, G_NAMES = make_composites(components)
y_cluster, centers_km = kmeans2_manual(Z_red)
y = y_cluster

print("=== Вариант m =", TARGET_M, "(итог lab3_report.tex / reduction_groups) ===")
print("Размеры кластеров:", np.bincount(y, minlength=2))
for i, comp in enumerate(components, start=1):
    print(f"  G{i}: {comp}")

if SPLIT_MODE == "stratified_y":
    order = np.argsort(y, kind="stable")
    split_desc = "стратификация по метке кластера y + чередование"
else:
    order = np.argsort(df["Glucose"].to_numpy(), kind="stable")
    split_desc = "вариационный ряд по Glucose + чередование"

work_idx = order[0::2]
ctrl_idx = order[1::2]
Z_work = Z_red[work_idx]
Z_ctrl = Z_red[ctrl_idx]
y_work = y[work_idx]
y_ctrl = y[ctrl_idx]

if BIN_MODE == "work_median":
    thr = np.median(Z_work, axis=0)
    Xb_work = (Z_work <= thr).astype(int)
    Xb_ctrl = (Z_ctrl <= thr).astype(int)
else:
    thr = Z_red.mean(axis=0)
    Xb_full = (Z_red <= thr).astype(int)
    Xb_work = Xb_full[work_idx]
    Xb_ctrl = Xb_full[ctrl_idx]

m = Xb_work.shape[1]
print("Пороги G1..G4:", dict(zip(G_NAMES, thr)))

full_lookup, full_default = build_sdnf(Xb_work, y_work)
label = apply_sdnf(Xb_ctrl, full_lookup, full_default)
acc_full = (label == y_ctrl).mean()
print(f"Полная модель на композитах (m={TARGET_M}): vs k-means на контроле: {acc_full:.4f}")
print("Уникальных векторов на рабочей:", len(full_lookup), "из", 2**m)

rows = []
for r in range(1, m + 1):
    for subset in combinations(range(m), r):
        mask = list(subset)
        lu, dc = build_sdnf(Xb_work, y_work, feat_mask=mask)
        pred = apply_sdnf(Xb_ctrl, lu, dc, feat_mask=mask)
        rows.append({
            "n": r,
            "features": "+".join(G_NAMES[i] for i in subset),
            "acc_vs_label": (pred == label).mean(),
            "acc_vs_true": (pred == y_ctrl).mean(),
            "lookup_size": len(lu),
        })

resdf_m4 = pd.DataFrame(rows).sort_values(["acc_vs_label", "n"], ascending=[False, True])
out_csv = f"tables/adequacy_composites_m{TARGET_M}.csv"
resdf_m4.to_csv(out_csv, index=False)
print("\nТоп-5 по acc_vs_label (m=4):")
print(resdf_m4.head(5).to_string(index=False))
print("\nСохранено:", out_csv)

outcome = df["Outcome"].to_numpy(dtype=int)

ks_rows = []
for i, gname in enumerate(G_NAMES):
    tr = Z_red[work_idx, i]
    te = Z_red[ctrl_idx, i]
    dn, dc = ks_stat(tr, te)
    ks_rows.append({
        "feature": gname,
        "D_n": round(dn, 4),
        "D_crit": round(dc, 4),
        "result": "ОК" if dn <= dc else "РАЗЛИЧАЮТСЯ",
    })
ks_df = pd.DataFrame(ks_rows)
ks_csv = f"tables/ks_composites_m{TARGET_M}.csv"
ks_df.to_csv(ks_csv, index=False, encoding="utf-8-sig")

train_bin_df = pd.DataFrame(Xb_work, columns=G_NAMES)
train_bin_df["Cluster"] = y_work
positive_train = train_bin_df[train_bin_df["Cluster"] == 1][G_NAMES]
unique_minterms = positive_train.drop_duplicates().values.tolist()
n_pos_train = int((y_work == 1).sum())

sdnf_terms = [minterm_to_str(m, G_NAMES) for m in unique_minterms]
sdnf_formula = " V\n    ".join(f"({t})" for t in sdnf_terms)

minimized = quine_mccluskey(unique_minterms, m)
min_terms_str = [minterm_to_str(list(impl), G_NAMES) for impl in minimized]
min_formula = " V\n    ".join(f"({t})" for t in min_terms_str)

train_logic_pred = train_bin_df[G_NAMES].apply(
    lambda row: evaluate_dnf(row, minimized, G_NAMES), axis=1
).values
train_acc = np.mean(train_logic_pred == y_work)
tr_tp = int(np.sum((train_logic_pred == 1) & (y_work == 1)))
tr_fn = int(np.sum((train_logic_pred == 0) & (y_work == 1)))
tr_fp = int(np.sum((train_logic_pred == 1) & (y_work == 0)))
tr_tn = int(np.sum((train_logic_pred == 0) & (y_work == 0)))

test_bin_df = pd.DataFrame(Xb_ctrl, columns=G_NAMES)
test_logic_pred = test_bin_df[G_NAMES].apply(
    lambda row: evaluate_dnf(row, minimized, G_NAMES), axis=1
).values
test_acc_vs_cluster = np.mean(test_logic_pred == y_ctrl)
test_tp = int(np.sum((test_logic_pred == 1) & (y_ctrl == 1)))
test_fn = int(np.sum((test_logic_pred == 0) & (y_ctrl == 1)))
test_fp = int(np.sum((test_logic_pred == 1) & (y_ctrl == 0)))
test_tn = int(np.sum((test_logic_pred == 0) & (y_ctrl == 0)))

outcome_ctrl = outcome[ctrl_idx]
cluster_vs_outcome_acc = np.mean(y_ctrl == outcome_ctrl)
logic_vs_outcome_acc = np.mean(test_logic_pred == outcome_ctrl)
agreement = test_acc_vs_cluster

if SPLIT_MODE == "variational_glucose":
    split_txt = "вариационный ряд по Glucose + чередование"
else:
    split_txt = "стратификация по метке кластера y + чередование"

bin_txt = (
    "1 если значение <= среднего, иначе 0"
    if BIN_MODE == "global_mean"
    else "1 если значение <= медианы рабочей выборки, иначе 0"
)

report_path = f"tables/logic_model_composites_m{TARGET_M}.txt"
sep = "=" * 60
with open(report_path, "w", encoding="utf-8") as f:
    f.write("ЛОГИЧЕСКАЯ МОДЕЛЬ (СДНФ и минимизация)\n")
    f.write(sep + "\n\n")
    f.write(
        "Вариант: m=4 (macro-F1 по Outcome); "
        "состав групп см. tables/reduction_groups.tex; здесь MST(|Y|) при TARGET_M=4.\n\n"
    )
    f.write(f"Признаки: {', '.join(G_NAMES)}\n")
    f.write(f"Бинаризация: {bin_txt}\n\n")
    f.write(f"Разбиение: {split_txt}\n")
    f.write(f"  Рабочая: {len(work_idx)}, Контрольная: {len(ctrl_idx)}\n\n")
    f.write("Проверка Колмогорова-Смирнова (alpha=0.05):\n")
    for _, row in ks_df.iterrows():
        f.write(
            f"  {row['feature']}: D_n={row['D_n']}, D_crit={row['D_crit']} -> {row['result']}\n"
        )
    f.write("\n")
    f.write("ОБУЧАЮЩАЯ ЧАСТЬ:\n")
    f.write(f"  Строк с Cluster=1: {n_pos_train}\n")
    f.write(f"  Уникальных минтермов: {len(unique_minterms)}\n\n")
    f.write(f"СДНФ ({len(unique_minterms)} минтермов):\n")
    f.write(f"  {sdnf_formula}\n\n")
    f.write(f"Минимизированная ДНФ ({len(minimized)} импликант):\n")
    f.write(f"  {min_formula}\n\n")
    f.write("ПРОВЕРКА НА ОБУЧАЮЩЕЙ ВЫБОРКЕ:\n")
    f.write(f"  Логическая модель vs кластеризация: {train_acc:.4f}\n\n")
    f.write("ПРОВЕРКА НА ТЕСТОВОЙ ВЫБОРКЕ:\n")
    f.write(f"  Логическая модель vs кластеризация: {test_acc_vs_cluster:.4f}\n")
    f.write(f"  TP={test_tp}, FN={test_fn}, FP={test_fp}, TN={test_tn}\n\n")
    f.write("СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТЕ vs Outcome:\n")
    f.write(f"  K-means (стохастическая):  {cluster_vs_outcome_acc:.4f}\n")
    f.write(f"  Логическая (минимиз. ДНФ): {logic_vs_outcome_acc:.4f}\n")
    f.write(f"  Совпадение моделей между собой: {agreement:.4f}\n")

print("\n" + "=" * 60)
print(f"ЛОГИЧЕСКАЯ МОДЕЛЬ, вариант m={TARGET_M}")
print("=" * 60)
print(ks_df.to_string(index=False))
print(f"\nСДНФ: {len(unique_minterms)} минтермов; ДНФ: {len(minimized)} импликант")
print(f"Train acc: {train_acc:.4f} | Test acc: {test_acc_vs_cluster:.4f}")
print(f"Отчёт: {report_path}")

=== Вариант m = 4 (итог lab3_report.tex / reduction_groups) ===
Размеры кластеров: [482 286]
  G1: ['Age', 'BloodPressure', 'Pregnancies']
  G2: ['BMI', 'Insulin', 'SkinThickness']
  G3: ['Glucose']
  G4: ['DiabetesPedigreeFunction']
Пороги G1..G4: {'G1': np.float64(4.5102810375396984e-17), 'G2': np.float64(1.0523989087592629e-16), 'G3': np.float64(-9.25185853854297e-18), 'G4': np.float64(2.4517425127138875e-16)}
Полная модель на композитах (m=4): vs k-means на контроле: 0.8620
Уникальных векторов на рабочей: 16 из 16

Топ-5 по acc_vs_label (m=4):
 n    features  acc_vs_label  acc_vs_true  lookup_size
 3    G2+G3+G4      1.000000     0.861979            8
 4 G1+G2+G3+G4      1.000000     0.861979           16
 3    G1+G2+G3      0.895833     0.872396            8
 1          G3      0.869792     0.851562            2
 2       G1+G3      0.869792     0.851562            4

Сохранено: tables/adequacy_composites_m4.csv

ЛОГИЧЕСКАЯ МОДЕЛЬ, вариант m=4
feature    D_n  D_crit result
     G1 